# News volatility prediction

Описание: Задача предсказания хаотичности и волатильности рынка во время выхода новости.

В качестве хаотичности будут использоваться следующие признаки:
- Range efficiency (in next N bars);
- Directional Noise Index (in next N bars);
- Range (in next N bars).

**Range efficiency** расчитывается как отношение разницы максимальной и минимальной цены (за N баров) к сумме всех диапазонов свечей (также за N баров).
Чем ближе значение к 1, тем более однонаправленным является движение. Чем ниже значение *RE*, тем больше внутренних движений произошло за N баров.

**Directional Noise Index** рассчитывается похожим образом, но вместо разницы максимальной и минимальной цены (за N баров), берется сумма абсолютных разниц между ценой закрытия и ценой открытия, которое делится на сумму всех диапазонов свечей (также за N баров). Значение близкое к 1 говорит о том, что движение является однонаправленным, а значение близкое к 0 говорит о том, что движение является хаотичным.


Фичи для ML модели:
- True Range (нормализовать)
- ATR (тоже нормализовать)
- ADX
- Close % change
- Volume % change
- Cumulative Volume % change
- Range efficiency *before news* (lags: 1, 3, 6, 12, 24)
- Directional noise index *before news* (lags: 1, 3, 6, 12, 24)
- Hurst exponent *before news* (lags: 1, 3, 6, 12, 24)

## Загрузка новостей

In [1]:
from typing import Literal

import pandas as pd
import numpy as np
import pytz
from scipy.stats import entropy

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

from warnings import filterwarnings
filterwarnings('ignore')

from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
moscow_tz = pytz.timezone('Europe/Moscow')
news_tz = pytz.timezone('Asia/Tehran')

In [3]:
PATH_TO_NEWS = "C:/Users/user/Documents/projects/ALGO/algotrading/dataset/economy_calendar/forex_factory_cache.csv"
news = pd.read_csv(PATH_TO_NEWS)
news.head()

,DateTime,Currency,Impact,Event,Actual,Forecast,Previous,Detail
0,2007-01-01T04:30:00+03:30,CNY,High Impact Expected,Manufacturing PMI,54.8,NaN,55.3,Source: CFLP (latest release) | Measures: Leve...
1,2007-01-01T23:59:59+03:30,CAD,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: Canadian banks will be closed in ...
2,2007-01-01T23:59:59+03:30,USD,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: US banks will be closed in observ...
3,2007-01-01T23:59:59+03:30,CNY,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: Chinese banks will be closed in o...
4,2007-01-01T23:59:59+03:30,EUR,Non-Economic,German Bank Holiday,NaN,NaN,NaN,FF Notice: The European Central Bank's Trans-E...


## Check Detail column

In [5]:
news['Detail'].to_list()[:10]

["Source: CFLP (latest release) | Measures: Level of a diffusion index based on surveyed purchasing managers in the manufacturing industry; | Usual Effect: 'Actual' greater than 'Forecast' is good for currency; | Frequency: Released monthly, on the last day of the current month; | Next Release: Feb 1, 2007 | FF Notes: Above 50.0 indicates industry expansion, below indicates contraction. Tends to have more impact when it's released ahead of the Caixin Manufacturing PMI because the reports are tightly correlated. Chinese data can have a broad impact on the currency markets due to China's influence on the global economy and investor sentiment; | Why Traders Care: It's a leading indicator of economic health - businesses react quickly to market conditions, and their purchasing managers hold perhaps the most current and relevant insight into the company's view of the economy; | Derived Via: Survey of 3,000 purchasing managers which asks respondents to rate the relative level of business cond

In [9]:
from collections import Counter
from tqdm import tqdm
import re

In [14]:
c = Counter()
for line in tqdm(news['Detail'].to_list()):
    try:
        words = re.findall(r"[A-Za-z][A-Za-z\s]*:", line)
        c.update(words)
    except Exception as e:
        pass

100%|██████████| 83427/83427 [00:08<00:00, 10211.58it/s]


In [12]:
c

Counter({'Next Release:': 83158,
         'Source:': 80867,
         'Usual Effect:': 80448,
         'Frequency:': 72614,
         'FF Notes:': 68617,
         'Why Traders Care:': 68115,
         'Measures:': 66678,
         'Acro Expand:': 47082,
         'Also Called:': 36975,
         'Derived Via:': 21886,
         'Description:': 10753,
         'Speaker:': 8434,
         'FF Notice:': 5179,
         'parts:': 145,
         'Monetary Policy:': 6,
         'Financial Stability:': 4,
         'Central Banking:': 4,
         'Central Banking in the Shadows:': 4,
         'At the heart of policy:': 4,
         'The euro at ten:': 3,
         'The Global Economy:': 3,
         'The Euro and the Dollar:': 2,
         'UK Monetary Policy:': 2,
         'Global Imbalances:': 2,
         'Economic Challenges:': 2,
         'The Challenges:': 2,
         'After the Crisis:': 2,
         'Central Bank statistics:': 2,
         's competitive imperative:': 2,
         'Fulfilling the Full E

In [44]:
main_details = ['Next Release',
                'Source',
                'Usual Effect',
                'Frequency',
                'FF Notes',
                'Why Traders Care',
                'Measures',
                'Acro Expand',
                'Also Called',
                'Derived Via',
                'Description',
                'Speaker',
                'FF Notice']

In [45]:
def extract_field_text_from_event(text: str, field: str):
    try:
        return re.findall(fr"{field}:\s*([^|]+)", text)[0]
    except Exception as e:
        return None

In [46]:
for main_detail in tqdm(main_details):
    news[main_detail] = news['Detail'].apply(lambda x: extract_field_text_from_event(x, main_detail))

100%|██████████| 13/13 [00:01<00:00,  9.38it/s]


In [81]:
news['Next Release'] = pd.to_datetime(news['Next Release'], errors='coerce')

In [87]:
news['utc_dt'] = pd.to_datetime(news['DateTime'], utc=True).dt.tz_convert(moscow_tz)
news['Next Release'] = pd.to_datetime(news['Next Release']).dt.tz_localize(moscow_tz)

In [95]:
news['Next Release Days'] = news['Next Release'] - news['utc_dt']
news['Next Release Days (Days)'] = news['Next Release Days'].dt.days
news['Next Release Days (Hours)'] = news['Next Release Days'].dt.total_seconds() / 3600

In [99]:
news.head()

,DateTime,Currency,Impact,Event,Actual,Forecast,Previous,Detail,Next Release,Source,...,class,event_type,is_key_event,event_weight,utc_dt,event_time,custom_event_time,Next Release Days,Next Release Days (Days),Next Release Days (Hours)
0,2007-01-01T04:30:00+03:30,CNY,High Impact Expected,Manufacturing PMI,54.8,NaN,55.3,Source: CFLP (latest release) | Measures: Leve...,2007-02-01 00:00:00+03:00,CFLP (latest release),...,ECONOMIC_ACTIVITY,PMI_MANUFACTURING,True,3,2007-01-01 04:00:00+03:00,2007-01-01 04:00:00+03:00,2007-01-01 04:00:00+03:00,30 days 20:00:00,30.0,740.000000
1,2007-01-01T23:59:59+03:30,CAD,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: Canadian banks will be closed in ...,2007-04-06 00:00:00+04:00,None,...,OTHER,OTHER,False,1,2007-01-01 23:29:59+03:00,2007-01-01 23:00:00+03:00,2007-01-01 23:30:00+03:00,93 days 23:30:01,93.0,2255.500278
2,2007-01-01T23:59:59+03:30,USD,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: US banks will be closed in observ...,2007-01-15 00:00:00+03:00,None,...,OTHER,OTHER,False,1,2007-01-01 23:29:59+03:00,2007-01-01 23:00:00+03:00,2007-01-01 23:30:00+03:00,13 days 00:30:01,13.0,312.500278
3,2007-01-01T23:59:59+03:30,CNY,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: Chinese banks will be closed in o...,2007-01-02 00:00:00+03:00,None,...,OTHER,OTHER,False,1,2007-01-01 23:29:59+03:00,2007-01-01 23:00:00+03:00,2007-01-01 23:30:00+03:00,0 days 00:30:01,0.0,0.500278
4,2007-01-01T23:59:59+03:30,EUR,Non-Economic,German Bank Holiday,NaN,NaN,NaN,FF Notice: The European Central Bank's Trans-E...,2007-04-06 00:00:00+04:00,None,...,OTHER,OTHER,False,1,2007-01-01 23:29:59+03:00,2007-01-01 23:00:00+03:00,2007-01-01 23:30:00+03:00,93 days 23:30:01,93.0,2255.500278


In [47]:
for col in main_details:
    print(f'======== {col} ========')
    display(news[col].value_counts()[:10])

======== Next Release ========


Next Release
Feb 15, 2024     45
Dec 16, 2021     45
Aug 15, 2024     43
Nov 15, 2024     43
Dec 14, 2017     43
Mar 1, 2012      42
Mar 1, 2017      42
Apr 28, 2017     42
Dec 16, 2020     42
Mar 1, 2019      42
Name: count, dtype: int64

======== Source ========


Source
S&P Global (latest release)                          6019
Statistics Canada (latest release)                   4010
Office for National Statistics (latest release)      3808
Bank of England (latest release)                     3255
Federal Reserve (latest release)                     3051
Census Bureau (latest release)                       2625
Bureau of Labor Statistics (latest release)          2557
Eurostat (latest release)                            2497
Bank of Japan (latest release)                       2380
Australian Bureau of Statistics (latest release)     2175
Name: count, dtype: int64

======== Usual Effect ========


...

Description
Chinese banks will be closed in observance of the Spring Festival;                                                           95
Chinese banks will be closed in observance of National Day;                                                                  83
Japanese banks will be closed in observance of the 4-day Bank Holiday;                                                       53
Due to hold a press conference, along with other MPC members, about the Inflation Report, in London;                         50
Due to hold a press conference about the coronavirus, in Washington DC;                                                      38
Chinese banks will be closed in observance of Labor Day;                                                                     34
Due to testify on the Semiannual Monetary Policy Report before the House Financial Services Committee, in Washington DC;     29
Due to testify on the Semiannual Monetary Policy Report before the Senate Banking Committee,

======== Speaker ========


Speaker
ECB President Jean-Claude Trichet;                             246
Federal Reserve Chairman Ben Bernanke;                         239
ECB President Christine Lagarde;                               223
Federal Reserve Bank of New York President John Williams;      218
BOJ Governor;                                                  203
ECB President Mario Draghi;                                    203
ECB President and Vice President;                              178
BOE Governor Mark Carney;                                      166
Federal Reserve Bank of New York President William Dudley;     162
Federal Reserve Chair Jerome Powell;                           159
Name: count, dtype: int64

======== FF Notice ========


FF Notice
Source released data 1 minute earlier than scheduled;                                                                                                                                                        438
Source released data 2 minutes later than scheduled;                                                                                                                                                         379
The European Central Bank's Trans-European Automated Real-time Gross Settlement Express Transfer (TARGET) system will be closed for this holiday, which tends to have a substantial impact on liquidity;     222
Source released data 2 minutes earlier than scheduled;                                                                                                                                                       221
For historical accuracy, this event was added to the calendar after its release time;                                                                     

### Standartize Actual, Forecast and Previous values

In [48]:
def standardize_economic_value(value):
    """
    Standardize economic indicator values from news data.
    Handles:
    - Numeric strings (e.g., '54.8', '55.3')
    - Percentages (e.g., '2.5%', '-1.2%')
    - Strings with units or text
    - NaN/None values
    - Empty strings
    
    Returns: float or np.nan
    """
    if pd.isna(value) or value == '' or value is None:
        return np.nan
    
    # Convert to string if not already
    value_str = str(value).strip()
    
    # Remove common prefixes/suffixes and clean
    value_str = value_str.replace(',', '')  # Remove thousand separators
    value_str = value_str.replace('$', '')   # Remove dollar signs
    value_str = value_str.replace('€', '')   # Remove euro signs
    value_str = value_str.replace('£', '')  # Remove pound signs
    
    # Handle percentages
    if '%' in value_str:
        value_str = value_str.replace('%', '')
        try:
            return float(value_str) / 100.0  # Convert percentage to decimal
        except (ValueError, TypeError):
            return np.nan
    
    # Try to convert to float directly
    try:
        return float(value_str)
    except (ValueError, TypeError):
        # If conversion fails, try to extract number from string
        import re
        # Extract first number found in the string
        numbers = re.findall(r'-?\d+\.?\d*', value_str)
        if numbers:
            try:
                return float(numbers[0])
            except (ValueError, TypeError):
                return np.nan
        return np.nan

In [49]:
# Standardize the columns
news['Actual_norm'] = news['Actual'].apply(standardize_economic_value)
news['Forecast_norm'] = news['Forecast'].apply(standardize_economic_value)
news['Previous_norm'] = news['Previous'].apply(standardize_economic_value)

In [50]:
news.tail()

,DateTime,Currency,Impact,Event,Actual,Forecast,Previous,Detail,Next Release,Source,...,Acro Expand,Also Called,Derived Via,Description,Speaker,FF Notice,Source2,Actual_norm,Forecast_norm,Previous_norm
83422,2025-04-07T10:30:00+04:30,CHF,Low Impact Expected,Foreign Currency Reserves,NaN,NaN,735B,Source: Swiss National Bank (latest release) |...,"May 7, 2025",Swiss National Bank (latest release),...,Swiss National Bank (SNB);,None,None,None,None,None,Swiss National Bank,NaN,NaN,735.000
83423,2025-04-07T12:00:00+04:30,EUR,Low Impact Expected,Sentix Investor Confidence,NaN,-8.9,-2.9,Source: Sentix (latest release) | Measures: Le...,"May 5, 2025",Sentix (latest release),...,None,None,"Survey of about 6,600 investors and analysts w...",None,None,None,Sentix,NaN,-8.900,-2.900
83424,2025-04-07T12:30:00+04:30,EUR,Low Impact Expected,Retail Sales m/m,NaN,0.5%,-0.3%,Source: Eurostat (latest release) | Measures: ...,"May 7, 2025",Eurostat (latest release),...,None,None,None,None,None,None,Eurostat,NaN,0.005,-0.003
83425,2025-04-07T18:00:00+04:30,CAD,Low Impact Expected,BOC Business Outlook Survey,NaN,NaN,NaN,Source: Bank of Canada (latest release) | Usua...,"Jul 21, 2025",Bank of Canada (latest release),...,Bank of Canada (BOC);,Senior Loan Officer Survey;,"Survey of about 1,000 businesses which asks re...",None,None,None,Bank of Canada,NaN,NaN,NaN
83426,2025-04-07T22:30:00+04:30,USD,Low Impact Expected,Consumer Credit m/m,NaN,15.2B,18.1B,Source: Federal Reserve (latest release) | Mea...,"May 7, 2025",Federal Reserve (latest release),...,None,None,None,None,None,None,Federal Reserve,NaN,15.200,18.100


### Установка ранга для поля Impact

In [51]:
def set_rank_for_impact(impact: str):
    if impact == 'High Impact Expected':
        return 3
    elif impact == 'Medium Impact Expected':
        return 2
    elif impact == 'Low Impact Expected':
        return 1
    else:
        return 0

In [52]:
news['impact_rank'] = news['Impact'].apply(set_rank_for_impact)

### Классификация новостей по группам

In [53]:
NEWS_CLASSES = [
    "MONETARY_POLICY",      # решения ЦБ
    "CB_SPEECH",            # речи ЦБ
    "INFLATION",            # инфляция
    "LABOR_MARKET",         # рынок труда
    "ECONOMIC_ACTIVITY",    # рост / производство
    "SENTIMENT",            # опросы / ожидания
    "CONSUMER_HOUSING",     # потребитель / жильё
    "TRADE_FINANCE",        # торговля / деньги / финансы
    "COMMODITIES"           # нефть / газ / сырьё
]

In [54]:
MONETARY_POLICY = [
    "rate", "policy", "monetary", "fomc", "mpc", "meeting",
    "minutes", "statement", "votes",
    "asset purchase", "purchase facility", "refinancing",
    "cash rate", "bank rate", "official cash",
    "fed", "ecb", "boe", "boj", "rba", "rbnz", "boc", "snb",
    "overnight rate", "federal funds", "funds rate"
]

CB_SPEECH = [
    "speaks", "press conference", "press",
    "chair", "president", "gov",
    "powell", "lagarde", "bailey", "kuroda", "draghi",
    "yellen", "carney", "bullard", "waller", "kashkari",
    "member"
]

INFLATION = [
    "cpi", "core cpi", "ppi", "core ppi",
    "pce", "core pce",
    "inflation", "price index", "prices",
    "median cpi", "trimmed cpi",
    "inflation expectations"
]

LABOR_MARKET = [
    "employment", "unemployment", "job", "claims",
    "non farm", "nonfarm", "payrolls",
    "adp", "jolts", "job openings",
    "earnings", "hourly earnings",
    "job cuts", "challenger"
]

ECONOMIC_ACTIVITY = [
    "gdp", "industrial", "production",
    "manufacturing", "services",
    "orders", "factory orders",
    "output", "capacity", "utilization",
    "productivity", "investment"
]

SENTIMENT = [
    "pmi", "ism", "confidence", "sentiment",
    "ifo", "zew", "gfk", "nfib", "philly",
    "survey", "optimism", "barometer",
    "expectations", "watchers"
]

CONSUMER_HOUSING = [
    "retail", "sales", "consumer",
    "housing", "home", "mortgage",
    "building", "permits", "starts",
    "house price", "hpi"
]

TRADE_FINANCE = [
    "trade", "balance", "current account",
    "budget", "borrowing", "credit",
    "money supply", "m2", "m3",
    "lending", "loans", "reserves",
    "foreign", "currency"
]

COMMODITIES = [
    "oil", "crude", "gas",
    "inventories", "storage",
    "opec", "commodity prices"
]

In [55]:
CLASS_KEYWORDS = {
    "MONETARY_POLICY": MONETARY_POLICY,
    "CB_SPEECH": CB_SPEECH,
    "INFLATION": INFLATION,
    "LABOR_MARKET": LABOR_MARKET,
    "ECONOMIC_ACTIVITY": ECONOMIC_ACTIVITY,
    "SENTIMENT": SENTIMENT,
    "CONSUMER_HOUSING": CONSUMER_HOUSING,
    "TRADE_FINANCE": TRADE_FINANCE,
    "COMMODITIES": COMMODITIES
}

In [57]:
def classify_news(title: str):
    title = title.lower()
    scores = {}

    for cls, keywords in CLASS_KEYWORDS.items():
        scores[cls] = sum(1 for k in keywords if k in title)

    if max(scores.values()) == 0:
        return "OTHER"

    return max(scores, key=scores.get)

In [58]:
news['class'] = news['Event'].astype(str).apply(classify_news)

### Добавление категорий отдельных новостей, важности новости и ее веса

In [59]:
def detect_event_type(title: str) -> str:
    t = title.lower()

    if "non farm" in t or "nonfarm" in t:
        return "NFP"
    if "cpi" in t and "core" in t:
        return "CORE_CPI"
    if "cpi" in t:
        return "CPI"
    if "pce" in t:
        return "PCE"
    if "fomc" in t and "rate" in t:
        return "FOMC_RATE"
    if "fomc" in t and "press" in t:
        return "FOMC_PRES_CONF"
    if "pmi" in t and "manufacturing" in t:
        return "PMI_MANUFACTURING"  # TODO: Check this line
    if "pmi" in t and "services" in t:
        return "PMI_SERVICES"
    if "gdp" in t:
        return "GDP"
    if "retail" in t:
        return "RETAIL_SALES"

    return "OTHER"

In [60]:
news["event_type"] = news['Event'].apply(detect_event_type)

In [61]:
KEY_EVENTS = {
    "NFP", "CPI", "CORE_CPI", "PCE",
    "FOMC_RATE", "FOMC_PRES_CONF",
    "PMI_MANUFACTURING", "PMI_SERVICES",
    "GDP"
}

news["is_key_event"] = news["event_type"].isin(KEY_EVENTS)

In [62]:
EVENT_WEIGHTS = {
    "NFP": 5,
    "CPI": 5,
    "CORE_CPI": 5,
    "PCE": 4,
    "FOMC_RATE": 5,
    "FOMC_PRES_CONF": 4,
    "PMI_MANUFACTURING": 3,
    "PMI_SERVICES": 3,
    "GDP": 3,
    "RETAIL_SALES": 2,
    "OTHER": 1
}

news["event_weight"] = news["event_type"].map(EVENT_WEIGHTS)

In [63]:
news.head(3)

,DateTime,Currency,Impact,Event,Actual,Forecast,Previous,Detail,Next Release,Source,...,Previous_norm,impact_rank,detail_description,detail_usual_effect,detail_next_release,detail_why_traders_care,class,event_type,is_key_event,event_weight
0,2007-01-01T04:30:00+03:30,CNY,High Impact Expected,Manufacturing PMI,54.8,NaN,55.3,Source: CFLP (latest release) | Measures: Leve...,"Feb 1, 2007",CFLP (latest release),...,55.3,3,None,'Actual' greater than 'Forecast' is good for c...,"Feb 1, 2007",It's a leading indicator of economic health - ...,ECONOMIC_ACTIVITY,PMI_MANUFACTURING,True,3
1,2007-01-01T23:59:59+03:30,CAD,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: Canadian banks will be closed in ...,"Apr 6, 2007",None,...,NaN,0,Canadian banks will be closed in observance of...,Low liquidity and irregular volatility,"Apr 6, 2007",Banks facilitate the majority of foreign excha...,OTHER,OTHER,False,1
2,2007-01-01T23:59:59+03:30,USD,Non-Economic,Bank Holiday,NaN,NaN,NaN,Description: US banks will be closed in observ...,"Jan 15, 2007",None,...,NaN,0,US banks will be closed in observance of New Y...,Low liquidity and irregular volatility,"Jan 15, 2007",Banks facilitate the majority of foreign excha...,OTHER,OTHER,False,1


### Построение агрегированных новостей для МЛ модели

In [64]:
news['utc_dt'] = pd.to_datetime(news['DateTime'], utc=True).dt.tz_convert(moscow_tz)
# news['event_hour'] = news['utc_dt'].dt.ceil('H', ambiguous='NaT', nonexistent='shift_forward')
news['event_time'] = news['utc_dt'].dt.floor('30min')


In [65]:
def floor_or_ceil(x: str):
    x_dt = pd.to_datetime(x)
    if (29 >= x_dt.minute >= 20) or (59 >= x_dt.minute >= 50):  # TODO: May be change to last 5 minutes (20 -> 25 and 50 -> 55)
        return x_dt.ceil('30min', ambiguous='NaT', nonexistent='shift_forward')
    else:
        return x_dt.floor('30min')

In [66]:
news['custom_event_time'] = news['utc_dt'].progress_apply(floor_or_ceil)

100%|██████████| 83427/83427 [00:05<00:00, 15572.15it/s]


In [67]:
def build_hour_features(
    news_df: pd.DataFrame,
    key_event_types: list[str] | None = None,
    event_time_column: str = 'custom_event_time'
) -> pd.DataFrame:
    """
    Build hour-level aggregated features from event-level news dataframe.

    Parameters
    ----------
    news_df : pd.DataFrame
        Event-level news data. Must contain:
        ['event_hour', 'event_type', 'news_class',
         'impact_rank', 'is_key_event', 'event_weight']

    key_event_types : list[str], optional
        List of key event types to build presence flags for
        (e.g. ['NFP', 'CPI', 'PMI_MANUFACTURING']).
        If None, inferred from is_key_event == True.

    Returns
    -------
    pd.DataFrame
        Hour-level feature table indexed by event_hour.
    """

    df = news_df.copy()

    # -----------------------------
    # 1. BASIC COUNTS & INTENSITY
    # -----------------------------
    agg = df.groupby(event_time_column).agg(
        news_count=("event_type", "count"),
        high_impact_count=("impact_rank", lambda x: (x == 3).sum()),
        key_event_count=("is_key_event", "sum"),
        sum_impact=("impact_rank", "sum"),
        sum_event_weight=("event_weight", "sum"),
        max_event_weight=("event_weight", "max"),
    )

    # -----------------------------
    # 2. DOMINANT EVENT TYPE
    # -----------------------------
    dominant_event = (
        df.groupby([event_time_column, "event_type"])
          .size()
          .reset_index(name="cnt")
          .sort_values([event_time_column, "cnt"], ascending=[True, False])
          .drop_duplicates(event_time_column)
          .set_index(event_time_column)[["event_type"]]
          .rename(columns={"event_type": "dominant_event_type"})
    )

    # -----------------------------
    # 3. EVENT ENTROPY (STRUCTURAL NOISE)
    # -----------------------------
    def event_entropy(x: pd.Series) -> float:
        probs = x.value_counts(normalize=True)
        return entropy(probs) if len(probs) > 1 else 0.0

    entropy_df = (
        df.groupby(event_time_column)["event_type"]
          .apply(event_entropy)
          .to_frame("event_entropy")
    )

    # -----------------------------
    # 4. KEY EVENT PRESENCE FLAGS
    # -----------------------------
    if key_event_types is None:
        key_event_types = (
            df.loc[df["is_key_event"], "event_type"]
              .dropna()
              .unique()
              .tolist()
        )

    presence_flags = []
    for evt in key_event_types:
        s = (
            df[df["event_type"] == evt]
            .groupby(event_time_column)
            .size()
            .rename(f"has_{evt.lower()}")
        )
        presence_flags.append(s)

    presence_df = (
        pd.concat(presence_flags, axis=1)
        .fillna(0)
        .astype(int)
    ) if presence_flags else pd.DataFrame(index=agg.index)

    # -----------------------------
    # 5. FINAL MERGE
    # -----------------------------
    hour_features = (
        agg
        .join(dominant_event, how="left")
        .join(entropy_df, how="left")
        .join(presence_df, how="left")
        .reset_index()
    )

    # Fill NaNs (safety)
    num_cols = hour_features.select_dtypes(include=[np.number]).columns
    hour_features[num_cols] = hour_features[num_cols].fillna(0)

    hour_features["dominant_event_type"] = (
        hour_features["dominant_event_type"]
        .fillna("NONE")
        .astype("category")
    )

    # Create field 'Last_key_event_bars_ago' (example last important event was N bars ago and it was NFP)
    # Find last key event name and how many hours ago it was for each hour row

    # Prepare a mask for rows with actual key events (using is_key_event)
    key_event_mask = hour_features['dominant_event_type'].isin(key_event_types)

    # Build up last key event name using a forward fill
    hour_features['last_key_event_name'] = (
        hour_features['dominant_event_type'].where(key_event_mask).shift(1)
        .ffill()
    )

    # For time calculations, get event_hour of previous key events
    last_key_event_time = hour_features[event_time_column].where(key_event_mask).shift(1).ffill()

    hour_features['last_key_event_hours_ago'] = (
        (hour_features[event_time_column] - last_key_event_time).dt.total_seconds() / 3600
    )

    return hour_features

In [68]:
news_agg = build_hour_features(news, key_event_types=KEY_EVENTS)

In [69]:
news_agg.sample(5)

,custom_event_time,news_count,high_impact_count,key_event_count,sum_impact,sum_event_weight,max_event_weight,dominant_event_type,event_entropy,has_core_cpi,has_nfp,has_pmi_services,has_cpi,has_fomc_pres_conf,has_pce,has_fomc_rate,has_gdp,has_pmi_manufacturing,last_key_event_name,last_key_event_hours_ago
8826,2009-12-21 13:00:00+03:00,1,0,0,1,1,1,OTHER,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CORE_CPI,94.0
31903,2017-08-29 09:30:00+03:00,1,0,0,1,1,1,OTHER,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,CORE_CPI,1.5
45567,2022-03-08 03:30:00+03:00,1,0,0,1,1,1,OTHER,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_SERVICES,106.0
14430,2011-10-31 00:30:00+04:00,6,0,1,8,10,5,OTHER,0.450561,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,PCE,56.0
45956,2022-04-25 11:00:00+03:00,1,0,0,2,1,1,OTHER,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_MANUFACTURING,66.5


In [70]:
news_agg[['custom_event_time', 'dominant_event_type', 'last_key_event_name', 'last_key_event_hours_ago']].head(10)

,custom_event_time,dominant_event_type,last_key_event_name,last_key_event_hours_ago
0,2007-01-01 04:00:00+03:00,PMI_MANUFACTURING,NaN,NaN
1,2007-01-01 23:30:00+03:00,OTHER,PMI_MANUFACTURING,19.5
2,2007-01-02 01:30:00+03:00,OTHER,PMI_MANUFACTURING,21.5
3,2007-01-02 03:00:00+03:00,OTHER,PMI_MANUFACTURING,23.0
4,2007-01-02 05:30:00+03:00,PMI_MANUFACTURING,PMI_MANUFACTURING,25.5
5,2007-01-02 08:30:00+03:00,OTHER,PMI_MANUFACTURING,3.0
6,2007-01-02 11:00:00+03:00,PMI_MANUFACTURING,PMI_MANUFACTURING,5.5
7,2007-01-02 11:30:00+03:00,PMI_MANUFACTURING,PMI_MANUFACTURING,0.5
8,2007-01-02 12:00:00+03:00,PMI_MANUFACTURING,PMI_MANUFACTURING,0.5
9,2007-01-02 12:30:00+03:00,PMI_MANUFACTURING,PMI_MANUFACTURING,0.5


### TODO
сделать агрегацию новостей по часу с указанием:
- сколько новостей собралось в этом часу
- как давно была последняя новость
- как давно была важная новость
- (дальше с присоединением котировок)
- какие параметры у новости были тогда в момент выхода прошлой новости

In [43]:
news_agg['last_key_event_name'].value_counts()

last_key_event_name
CPI                  15788
GDP                  12280
PMI_MANUFACTURING     9656
CORE_CPI              8088
PMI_SERVICES          7895
PCE                    481
FOMC_PRES_CONF         477
NFP                    302
OTHER                    0
RETAIL_SALES             0
Name: count, dtype: int64

## Загрузка котировок

In [ ]:
# import MetaTrader5 as mt5
# import numpy as np
# import pandas as pd
# from datetime import datetime

In [ ]:
# def get_prices_from_mt5(ticker: str, tf=mt5.TIMEFRAME_D1, from_date: datetime = datetime(2010, 1, 1), till_date: datetime = datetime.now()):
#     # connect to MetaTrader 5
#     path_to_terminal = "C:\\Program Files\\AMarkets - MetaTrader 5\\terminal64.exe"
#     if not mt5.initialize(path=path_to_terminal):
#         print("initialize() failed")
#         mt5.shutdown()

#     rates = mt5.copy_rates_range(ticker, tf, from_date, till_date)
#     df = pd.DataFrame(rates)
#     df['time'] = pd.to_datetime(df['time'], unit='s')
#     df = df.drop(['real_volume'], axis=1)
#     for col in ['open', 'high', 'low', 'close']:
#         df[col] = df[col].apply(lambda x: np.round(x, 5))
#     return df

In [ ]:
# prices = get_prices_from_mt5('EURUSDb', mt5.TIMEFRAME_H1, from_date=datetime(2014, 1, 1), till_date=datetime(2025, 4, 10))

In [1]:
import os

In [ ]:
def read_pair(pair: str, path2folder: str = "..\\dataset\\fx_data\\"):
    return pd.read_csv('..\\dataset\\fx_data\\EURUSD_M30.csv',
                        sep="\t",
                        names=["time", "open", "high", "low", "close", "volume", "spread"],
                        header=0,
                        parse_dates=["time"])

In [65]:
prices = pd.read_csv('..\\dataset\\fx_data\\EURUSD_M30.csv',
    sep="\t",
    names=["time", "open", "high", "low", "close", "volume", "spread"],
    header=0,
    parse_dates=["time"])
prices.head()

,time,open,high,low,close,volume,spread
0,2009-12-31 16:30:00,1.43989,1.44022,1.43706,1.43972,2045,9
1,2009-12-31 17:00:00,1.43977,1.44037,1.43817,1.43838,2203,9
2,2009-12-31 17:30:00,1.43831,1.43899,1.43612,1.43690,1903,9
3,2009-12-31 18:00:00,1.43669,1.43697,1.43154,1.43206,2148,9
4,2009-12-31 18:30:00,1.43189,1.43551,1.43025,1.43465,2080,8


In [67]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199422 entries, 0 to 199421
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   time    199422 non-null  datetime64[ns]
 1   open    199422 non-null  float64       
 2   high    199422 non-null  float64       
 3   low     199422 non-null  float64       
 4   close   199422 non-null  float64       
 5   volume  199422 non-null  int64         
 6   spread  199422 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(2)
memory usage: 10.7 MB


In [66]:
print('News min date: ', pd.to_datetime(prices['time']).min())
print('News max date: ', pd.to_datetime(prices['time']).max())

News min date:  2009-12-31 16:30:00
News max date:  2026-01-14 10:30:00


## Фичи для котировок

### Фичи для определения волатильности
- range efficiency
- directional noise index
- normalized rolling volume
- normalized rolling range

In [55]:
import talib as ta
from hurst import compute_Hc
from tqdm.auto import tqdm

tqdm.pandas(desc="Processing DataFrame")

In [56]:
PERIOD = 12

In [68]:
def kaufman_efficiency_ratio(data: pd.DataFrame, window: int):
    direction = data.close.diff().abs()
    sum_range = data.range.rolling(window).sum()
    return direction / sum_range

In [69]:
def custom_range_efficiency(data: pd.DataFrame, window: int = 3) -> float:
    """
    Calculates the range efficiency for a given window size on OHLC price data.
    
    Range efficiency is defined as the ratio of the total range (difference between
    highest high and lowest low over the window) to the sum of the ranges (high-low per bar)
    within that window. 
    Higher values indicate more directional movement within the window,
    while lower values indicate more choppy or noisy price action.
    """
    total_range = data.high.rolling(window).max() - data.low.rolling(window).min()
    sum_range = data.range.rolling(window=window).sum()
    return total_range / sum_range

In [70]:
def noise_inside_the_bars(data: pd.DataFrame):
    body = (data['open'] - data['close']).abs()
    bar_range = data['high'] - data['low']
    return (bar_range - body) / (body + 1e-9)

In [71]:
def relative_atr(data: pd.DataFrame, window: int):
    atr = ta.ATR(data['high'], data['low'], data['close'], window)
    return (atr - atr.rolling(window*5).mean()) / atr.rolling(window*5).std()

In [72]:
def get_bb_width(data: pd.DataFrame, window: int):
    upper_band, middle_band, lower_band = ta.BBANDS(
        data['close'], 
        timeperiod=window, 
        nbdevup=3,
        nbdevdn=3,
        matype=0
    )
    return upper_band - lower_band

In [73]:
def rolling_hurst(series: pd.Series, window=100):
    def get_h(sub_series):
        if len(sub_series) < 50:
            return np.nan
        H, _, _ = compute_Hc(sub_series, kind='price', simplified=True)
        return H
    
    return series.rolling(window=window).progress_apply(get_h, raw=False)

In [78]:
def calculate_chaos_features(data: pd.DataFrame, window: int = 14):
    data['range'] = ta.TRANGE(data['high'], data['low'], data['close'])

    # 1. Kaufman Efficiency Ratio
    data['kaufman_efficiency_ratio'] = kaufman_efficiency_ratio(data, window=window)

    # 2. Custom Range efficiency
    data['custom_range_efficiency'] = custom_range_efficiency(data, window=window)

    # 3. Wick-to-Body Ratio (Шум внутри баров)
    data['wick_ratio'] = noise_inside_the_bars(data)

    # 4. Relative ATR
    data['relative_atr'] = relative_atr(data, window=window)

    # 5. BB width
    data['bb_width'] = get_bb_width(data, window=window)

    # 6. Anomaly volume
    v_mean = data['volume'].rolling(window*5).mean()
    v_std = data['volume'].rolling(window*5).std()
    data['volume_zscore'] = (data['volume'] - v_mean) / (v_std + 1e-9)

    # 7. Price to Volume
    data['price_volume_ratio'] = data.range / (data.volume + 1e-9)

    # 8. Hurst Exponent
    data['hurst'] = rolling_hurst(data['close'], window=100)
    
    return data

In [79]:
prices = calculate_chaos_features(prices, window=PERIOD)

Processing DataFrame: 199323it [01:54, 1745.01it/s]


In [83]:
def label_future_chaos(data: pd.DataFrame, N=5, stop_mult=1.5, neutral_threshold: float = 0.3):
    """
    Генерирует сигналы о будущем хаосе на N баров вперед.
    stop_mult: множитель ATR для определения "выноса стопов"
    """
    atr = ta.ATR(data['high'], data['low'], data['close'], N*3)
    
    # Future min/max values
    future_max = data['high'].shift(-N).rolling(N).max()
    future_min = data['low'].shift(-N).rolling(N).min()
    future_range = future_max - future_min
    future_close = data['close'].shift(-N)
    current_close = data['close']

    # Find stop levels
    upper_stop = current_close + (atr * stop_mult)
    lower_stop = current_close - (atr * stop_mult)

    # Targets
    target_volatility_expantion = future_range > (atr * 3)
    target_whipsaw = (future_max > upper_stop) & (future_min < lower_stop)
    target_chaos = target_volatility_expantion | target_whipsaw

    # Return close price to open
    returned = abs(future_close - current_close) < (atr * neutral_threshold)
    target_neutral_chaos = (target_whipsaw & returned).astype(int)
    target_strong_move = (target_whipsaw & ~returned).astype(int)

    # data['target_whipsaw'] = target_whipsaw
    data['target_volatility_expantion'] = target_volatility_expantion.astype(int)
    data['target_chaos'] = target_chaos.astype(int)
    data['target_neutral_chaos'] = target_neutral_chaos
    data['target_strong_move'] = target_strong_move

    return data

In [84]:
prices = label_future_chaos(prices, N=12)

In [85]:
prices.sample(5)

,time,open,high,low,close,volume,spread,range,kaufman_efficiency_ratio,custom_range_efficiency,wick_ratio,relative_atr,bb_width,volume_zscore,price_volume_ratio,hurst,target_volatility_expantion,target_chaos,target_neutral_chaos,target_strong_move
158782,2022-10-06 05:00:00,0.99105,0.99181,0.99061,0.99091,12415,6,0.00120,0.011696,0.444444,7.571374,-0.855597,0.008514,0.359102,9.665727e-08,0.675524,1,1,0,0
34902,2012-10-17 17:30:00,1.31242,1.31282,1.31176,1.31211,7441,5,0.00106,0.016885,0.232571,2.419347,0.656491,0.004476,1.251011,1.424540e-07,0.770749,0,0,0,0
192086,2025-06-11 14:00:00,1.14383,1.14404,1.14281,1.14297,1725,4,0.00123,0.063150,0.300892,0.430232,-0.296985,0.006310,-0.493799,7.130435e-07,0.377824,1,1,0,0
38956,2013-02-14 18:00:00,1.33425,1.33445,1.33196,1.33424,6471,5,0.00249,0.000431,0.293966,247.975202,1.046595,0.005900,1.314921,3.847937e-07,0.690175,0,0,0,0
132317,2020-08-20 19:30:00,1.18564,1.18589,1.18526,1.18541,5137,3,0.00063,0.011917,0.314796,1.739123,0.707816,0.007882,-0.391761,1.226397e-07,0.666390,0,0,0,0


## Присоединение к котировочным фичам новостные фичи

In [86]:
prices.head()

,time,open,high,low,close,volume,spread,range,kaufman_efficiency_ratio,custom_range_efficiency,wick_ratio,relative_atr,bb_width,volume_zscore,price_volume_ratio,hurst,target_volatility_expantion,target_chaos,target_neutral_chaos,target_strong_move
0,2009-12-31 16:30:00,1.43989,1.44022,1.43706,1.43972,2045,9,NaN,NaN,NaN,17.588132,NaN,NaN,NaN,NaN,NaN,0,0,0,0
1,2009-12-31 17:00:00,1.43977,1.44037,1.43817,1.43838,2203,9,0.00220,NaN,NaN,0.582733,NaN,NaN,NaN,9.986382e-07,NaN,0,0,0,0
2,2009-12-31 17:30:00,1.43831,1.43899,1.43612,1.43690,1903,9,0.00287,NaN,NaN,1.035460,NaN,NaN,NaN,1.508145e-06,NaN,0,0,0,0
3,2009-12-31 18:00:00,1.43669,1.43697,1.43154,1.43206,2148,9,0.00543,NaN,NaN,0.172786,NaN,NaN,NaN,2.527933e-06,NaN,0,0,0,0
4,2009-12-31 18:30:00,1.43189,1.43551,1.43025,1.43465,2080,8,0.00526,NaN,NaN,0.905797,NaN,NaN,NaN,2.528846e-06,NaN,0,0,0,0


In [87]:
prices['utc_dt'] = pd.to_datetime(prices['time']).dt.tz_localize(moscow_tz)

In [92]:
df = prices.merge(news_agg, how='left', left_on='utc_dt', right_on='custom_event_time').dropna()

In [93]:
df.head()

,time,open,high,low,close,volume,spread,range,kaufman_efficiency_ratio,custom_range_efficiency,...,has_cpi,has_fomc_pres_conf,has_nfp,has_gdp,has_fomc_rate,has_pmi_services,has_core_cpi,has_pmi_manufacturing,last_key_event_name,last_key_event_hours_ago
103,2010-01-04 21:00:00,1.44127,1.44221,1.44105,1.44199,1195,9,0.00116,0.019868,0.248519,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_MANUFACTURING,3.0
108,2010-01-04 23:30:00,1.44188,1.44215,1.44074,1.44105,1725,9,0.00141,0.032807,0.228821,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_MANUFACTURING,5.5
115,2010-01-05 03:00:00,1.44238,1.44395,1.44181,1.44253,635,9,0.00214,0.004992,0.218525,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_MANUFACTURING,9.0
131,2010-01-05 11:00:00,1.44508,1.44535,1.44183,1.44250,2112,9,0.00352,0.095450,0.293378,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_MANUFACTURING,17.0
133,2010-01-05 12:00:00,1.44191,1.44368,1.44126,1.44274,2017,8,0.00242,0.035064,0.275696,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,PMI_MANUFACTURING,18.0


In [100]:
def add_time_feaures(data: pd.DataFrame, dt_col: str):
    data[dt_col] = pd.to_datetime(data[dt_col])
    data['quarter'] = data[dt_col].dt.quarter
    data['week'] = data[dt_col].dt.isocalendar().week
    data['dayofweek'] = data[dt_col].dt.dayofweek
    data['day'] = data[dt_col].dt.day
    data['hour'] = data[dt_col].dt.hour
    data['minute'] = data[dt_col].dt.minute
    return data

In [101]:
df = add_time_feaures(df, 'time')

In [102]:
df.head()

,time,open,high,low,close,volume,spread,range,kaufman_efficiency_ratio,custom_range_efficiency,...,has_core_cpi,has_pmi_manufacturing,last_key_event_name,last_key_event_hours_ago,quarter,week,dayofweek,day,hour,minute
103,2010-01-04 21:00:00,1.44127,1.44221,1.44105,1.44199,1195,9,0.00116,0.019868,0.248519,...,0.0,0.0,PMI_MANUFACTURING,3.0,1,1,0,4,21,0
108,2010-01-04 23:30:00,1.44188,1.44215,1.44074,1.44105,1725,9,0.00141,0.032807,0.228821,...,0.0,0.0,PMI_MANUFACTURING,5.5,1,1,0,4,23,30
115,2010-01-05 03:00:00,1.44238,1.44395,1.44181,1.44253,635,9,0.00214,0.004992,0.218525,...,0.0,0.0,PMI_MANUFACTURING,9.0,1,1,1,5,3,0
131,2010-01-05 11:00:00,1.44508,1.44535,1.44183,1.44250,2112,9,0.00352,0.095450,0.293378,...,0.0,0.0,PMI_MANUFACTURING,17.0,1,1,1,5,11,0
133,2010-01-05 12:00:00,1.44191,1.44368,1.44126,1.44274,2017,8,0.00242,0.035064,0.275696,...,0.0,0.0,PMI_MANUFACTURING,18.0,1,1,1,5,12,0


In [103]:
df.columns

Index(['time', 'open', 'high', 'low', 'close', 'volume', 'spread', 'range',
       'kaufman_efficiency_ratio', 'custom_range_efficiency', 'wick_ratio',
       'relative_atr', 'bb_width', 'volume_zscore', 'price_volume_ratio',
       'hurst', 'target_volatility_expantion', 'target_chaos',
       'target_neutral_chaos', 'target_strong_move', 'utc_dt',
       'custom_event_time', 'news_count', 'high_impact_count',
       'key_event_count', 'sum_impact', 'sum_event_weight', 'max_event_weight',
       'dominant_event_type', 'event_entropy', 'has_pce', 'has_cpi',
       'has_fomc_pres_conf', 'has_nfp', 'has_gdp', 'has_fomc_rate',
       'has_pmi_services', 'has_core_cpi', 'has_pmi_manufacturing',
       'last_key_event_name', 'last_key_event_hours_ago', 'quarter', 'week',
       'dayofweek', 'day', 'hour', 'minute'],
      dtype='object')

# ML

In [104]:
X = df[['utc_dt', 'quarter', 'week', 'dayofweek', 'day', 'hour', 'minute', 
       'range', 'kaufman_efficiency_ratio', 'custom_range_efficiency', 'wick_ratio',
       'relative_atr', 'bb_width', 'volume_zscore', 'price_volume_ratio',
       'hurst', 'custom_event_time', 'news_count', 'high_impact_count',
       'key_event_count', 'sum_impact', 'sum_event_weight', 'max_event_weight',
       'dominant_event_type', 'event_entropy', 'has_pce', 'has_cpi',
       'has_fomc_pres_conf', 'has_nfp', 'has_gdp', 'has_fomc_rate',
       'has_pmi_services', 'has_core_cpi', 'has_pmi_manufacturing',
       'last_key_event_name', 'last_key_event_hours_ago']]

y = df['target_volatility_expantion']

In [105]:
from catboost import CatBoostClassifier

In [108]:
# Prepare data for CatBoost
# Drop datetime columns as they can't be used directly as features
X_processed = X.drop(['utc_dt', 'custom_event_time'], axis=1)

# Identify categorical features
categorical_features = ['dominant_event_type', 'last_key_event_name']
categorical_indices = [X_processed.columns.get_loc(col) for col in categorical_features if col in X_processed.columns]

# Initialize and fit CatBoost model
model = CatBoostClassifier(
    iterations=1000,
    # task_type="GPU",
    # devices='0',
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=True,
    cat_features=categorical_indices
)

# Fit the model
model.fit(X_processed, y, cat_features=categorical_indices)

print("Model training completed!")

0:	total: 51.6ms	remaining: 51.6s
1:	total: 95.8ms	remaining: 47.8s
2:	total: 127ms	remaining: 42.2s
3:	total: 164ms	remaining: 40.9s
4:	total: 201ms	remaining: 40s
5:	total: 239ms	remaining: 39.6s
6:	total: 280ms	remaining: 39.7s
7:	total: 320ms	remaining: 39.7s
8:	total: 358ms	remaining: 39.5s
9:	total: 395ms	remaining: 39.1s
10:	total: 428ms	remaining: 38.5s
11:	total: 465ms	remaining: 38.3s
12:	total: 501ms	remaining: 38s
13:	total: 535ms	remaining: 37.7s
14:	total: 571ms	remaining: 37.5s
15:	total: 608ms	remaining: 37.4s
16:	total: 641ms	remaining: 37.1s
17:	total: 675ms	remaining: 36.8s
18:	total: 710ms	remaining: 36.6s
19:	total: 747ms	remaining: 36.6s
20:	total: 780ms	remaining: 36.3s
21:	total: 817ms	remaining: 36.3s
22:	total: 850ms	remaining: 36.1s
23:	total: 884ms	remaining: 36s
24:	total: 917ms	remaining: 35.8s
25:	total: 952ms	remaining: 35.7s
26:	total: 986ms	remaining: 35.5s
27:	total: 1.02s	remaining: 35.5s
28:	total: 1.06s	remaining: 35.5s
29:	total: 1.1s	remaining: 3

In [109]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

In [110]:
# Prepare data for CatBoost
# Drop datetime columns as they can't be used directly as features
X_processed = X.drop(['utc_dt', 'custom_event_time'], axis=1)

# Identify categorical features
categorical_features = ['dominant_event_type', 'last_key_event_name']
categorical_indices = [X_processed.columns.get_loc(col) for col in categorical_features if col in X_processed.columns]

# TimeSeriesSplit for time series cross-validation
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

In [111]:
# Store results for each fold
fold_results = []

print(f"Starting TimeSeriesSplit cross-validation with {n_splits} folds...")
print("=" * 80)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_processed), 1):
    print(f"\nFold {fold}/{n_splits}")
    print(f"Train size: {len(train_idx)}, Validation size: {len(val_idx)}")
    
    # Split data
    X_train, X_val = X_processed.iloc[train_idx], X_processed.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Initialize CatBoost model
    model = CatBoostClassifier(
        iterations=1000,
        # task_type="GPU",
        # devices='0',
        learning_rate=0.1,
        depth=6,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=42,
        verbose=100,
        cat_features=categorical_indices,
        early_stopping_rounds=50
    )
    
    # Fit the model with validation set (this will show metrics during training)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=categorical_indices,
        use_best_model=True,
        plot=False
    )
    
    # Predictions on validation set
    y_val_pred = model.predict(X_val)
    y_val_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)
    auc = roc_auc_score(y_val, y_val_pred_proba)
    
    fold_results.append({
        'fold': fold,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    })
    
    print(f"\nValidation Metrics for Fold {fold}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  AUC:       {auc:.4f}")
    print("-" * 80)

Starting TimeSeriesSplit cross-validation with 5 folds...

Fold 1/5
Train size: 7427, Validation size: 7425
0:	test: 0.8214997	best: 0.8214997 (0)	total: 18.4ms	remaining: 18.3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8639022559
bestIteration = 31

Shrink model to first 32 iterations.

Validation Metrics for Fold 1:
  Accuracy:  0.7957
  Precision: 0.8249
  Recall:    0.8161
  F1-Score:  0.8205
  AUC:       0.8639
--------------------------------------------------------------------------------

Fold 2/5
Train size: 14852, Validation size: 7425
0:	test: 0.8087263	best: 0.8087263 (0)	total: 23.9ms	remaining: 23.9s
100:	test: 0.8669051	best: 0.8670122 (95)	total: 2.73s	remaining: 24.3s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8670121508
bestIteration = 95

Shrink model to first 96 iterations.

Validation Metrics for Fold 2:
  Accuracy:  0.8004
  Precision: 0.8380
  Recall:    0.8317
  F1-Score:  0.8348
  AUC:       0.8670
------------

In [112]:
# Summary of all folds
print("\n" + "=" * 80)
print("CROSS-VALIDATION SUMMARY")
print("=" * 80)

results_df = pd.DataFrame(fold_results)
print("\nMetrics per fold:")
print(results_df.to_string(index=False))

print("\nAverage metrics across all folds:")
print(f"  Accuracy:  {results_df['accuracy'].mean():.4f} (+/- {results_df['accuracy'].std():.4f})")
print(f"  Precision: {results_df['precision'].mean():.4f} (+/- {results_df['precision'].std():.4f})")
print(f"  Recall:    {results_df['recall'].mean():.4f} (+/- {results_df['recall'].std():.4f})")
print(f"  F1-Score:  {results_df['f1'].mean():.4f} (+/- {results_df['f1'].std():.4f})")
print(f"  AUC:       {results_df['auc'].mean():.4f} (+/- {results_df['auc'].std():.4f})")

# Train final model on all data (optional - for production use)
print("\n" + "=" * 80)
print("Training final model on all data...")
print("=" * 80)

final_model = CatBoostClassifier(
    iterations=1000,
    # task_type="GPU",
    # devices='0',
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=100,
    cat_features=categorical_indices
)

final_model.fit(X_processed, y, cat_features=categorical_indices)

# Final predictions on full dataset
y_pred_final = final_model.predict(X_processed)
y_pred_proba_final = final_model.predict_proba(X_processed)[:, 1]

# Final metrics
final_accuracy = accuracy_score(y, y_pred_final)
final_auc = roc_auc_score(y, y_pred_proba_final)

print(f"\nFinal Model Metrics (on full dataset):")
print(f"  Accuracy:  {final_accuracy:.4f}")
print(f"  AUC:       {final_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y, y_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y, y_pred_final))

print("\nModel training completed!")


CROSS-VALIDATION SUMMARY

Metrics per fold:
 fold  accuracy  precision   recall       f1      auc
    1  0.795690   0.824887 0.816149 0.820495 0.863902
    2  0.800404   0.837995 0.831668 0.834819 0.867012
    3  0.821010   0.850816 0.851769 0.851292 0.883825
    4  0.807138   0.830897 0.852856 0.841733 0.872004
    5  0.845522   0.868142 0.876872 0.872485 0.898283

Average metrics across all folds:
  Accuracy:  0.8140 (+/- 0.0201)
  Precision: 0.8425 (+/- 0.0173)
  Recall:    0.8459 (+/- 0.0231)
  F1-Score:  0.8442 (+/- 0.0194)
  AUC:       0.8770 (+/- 0.0141)

Training final model on all data...
0:	total: 30.1ms	remaining: 30.1s
100:	total: 3.51s	remaining: 31.3s
200:	total: 7.06s	remaining: 28.1s
300:	total: 10.5s	remaining: 24.5s
400:	total: 14s	remaining: 20.9s
500:	total: 17.5s	remaining: 17.5s
600:	total: 21.1s	remaining: 14s
700:	total: 24.7s	remaining: 10.5s
800:	total: 28.2s	remaining: 7s
900:	total: 31.8s	remaining: 3.49s
999:	total: 35.3s	remaining: 0us

Final Model Metric